In [204]:
import pandas as pd

In [205]:
df = pd.read_csv("social_media_productivity_6000.csv")

In [206]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   age                    5880 non-null   float64
 1   daily_screen_time      5880 non-null   float64
 2   social_media_hours     5880 non-null   float64
 3   study_hours            5880 non-null   float64
 4   sleep_hours            5880 non-null   float64
 5   notifications_per_day  5880 non-null   float64
 6   focus_score            5880 non-null   float64
 7   addiction_level        5880 non-null   object 
 8   productivity_score     5880 non-null   float64
dtypes: float64(8), object(1)
memory usage: 422.0+ KB


In [207]:
df.sample(10)

,age,daily_screen_time,social_media_hours,study_hours,sleep_hours,notifications_per_day,focus_score,addiction_level,productivity_score
4840,18.0,10.06,5.48,4.41,7.08,64.0,100.00,High,29.49
4712,23.0,10.85,6.27,2.69,5.35,79.0,86.38,High,0.00
4488,30.0,2.52,2.08,3.38,5.82,207.0,100.00,Medium,63.01
2872,32.0,7.70,4.56,5.30,4.64,179.0,100.00,Medium,29.86
2529,17.0,10.57,4.63,0.08,8.51,77.0,89.08,Medium,15.43
1783,25.0,5.52,2.24,6.68,6.94,222.0,100.00,Medium,76.13
4752,21.0,7.03,3.67,4.83,8.17,260.0,100.00,Medium,40.22
5835,34.0,3.73,2.09,6.02,6.43,280.0,100.00,Medium,75.38
3322,20.0,7.05,2.42,4.33,5.14,272.0,97.74,Medium,42.78
248,17.0,3.43,1.22,2.38,8.81,143.0,100.00,Low,64.13


In [208]:
df["addiction_level"].value_counts()

addiction_level
Medium    3064
High      1857
Low        959
Name: count, dtype: int64

## **Handling missing values**

In [209]:
df.isnull().any()

age                      True
daily_screen_time        True
social_media_hours       True
study_hours              True
sleep_hours              True
notifications_per_day    True
focus_score              True
addiction_level          True
productivity_score       True
dtype: bool

In [210]:
def missing_values(df):
    for col in df.columns:
        if df[col].isnull().any():
            if df[col].dtype == "object":
                df[col].fillna(df[col].mode()[0], inplace=True)
            else:
                df[col].fillna(df[col].mean(), inplace=True)
    return df


In [211]:
df = missing_values(df)

## **Encoding**
There is only one column that has "object" values and we need to encode this column.

In [212]:
addiction_level_mapping = {
    "Low" : 0,
    "Medium" : 1,
    "High" : 2
}

df["addiction_level"] = df["addiction_level"].map(addiction_level_mapping)

## **Scaling**

In [267]:
from sklearn.preprocessing import MinMaxScaler

In [268]:
def scaling(df):
    scaler = MinMaxScaler()
    num_col = df.select_dtypes(include=['int64','float64', 'int32']).columns.drop("addiction_level")
    df[num_col] = scaler.fit_transform(df[num_col])
    return df
    

In [269]:
df = scaling(df)

In [270]:
df.sample(10)

,age,daily_screen_time,social_media_hours,study_hours,sleep_hours,notifications_per_day,focus_score,addiction_level,productivity_score
3934,0.250000,0.115115,0.179,0.24750,0.510,0.501792,1.000000,1,0.226600
1381,0.416667,0.967968,0.787,0.59500,0.098,0.469534,0.930879,2,0.000000
1973,0.875000,0.017017,0.086,0.34125,0.828,0.146953,1.000000,0,0.685000
4911,0.416667,0.858859,0.858,0.05750,0.524,0.849462,0.394792,2,0.000000
1087,0.958333,0.635636,0.536,0.20250,0.284,0.025090,0.881391,2,0.003200
3575,0.666667,0.708709,0.690,0.43125,0.232,0.752688,0.452766,2,0.000000
766,0.000000,0.329329,0.234,0.82750,0.746,0.154122,1.000000,1,0.673900
4343,0.625000,0.558559,0.559,0.34500,0.014,0.903226,0.741114,2,0.000000
2408,0.666667,0.844845,0.668,0.04250,0.252,0.939068,0.291009,2,0.000000
4502,0.958333,0.397397,0.386,0.79000,0.448,0.430108,1.000000,1,0.376141


## **Logistic Regression**

In [276]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

x = df.drop("addiction_level", axis=1)
y = df["addiction_level"]

x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(x_train, y_train)
y_pred = log_reg.predict(x_test)
class_rep = classification_report(y_test, y_pred)
print(class_rep)


              precision    recall  f1-score   support

           0       0.87      0.78      0.82       199
           1       0.89      0.93      0.91       651
           2       0.94      0.91      0.92       350

    accuracy                           0.90      1200
   macro avg       0.90      0.87      0.89      1200
weighted avg       0.90      0.90      0.90      1200



## **Decision Tree Classifier**

In [281]:
from sklearn.tree import DecisionTreeClassifier

dtc = DecisionTreeClassifier()

dtc.fit(x_train, y_train)

y_pred = dtc.predict(x_test)

cl_report = classification_report(y_test, y_pred)
print(cl_report)

              precision    recall  f1-score   support

           0       0.97      0.97      0.97       199
           1       0.98      0.96      0.97       651
           2       0.95      0.97      0.96       350

    accuracy                           0.97      1200
   macro avg       0.97      0.97      0.97      1200
weighted avg       0.97      0.97      0.97      1200



## **Random Forest Classifier**

In [286]:
from sklearn.ensemble import RandomForestClassifier

rfc = RandomForestClassifier()

rfc.fit(x_train, y_train)

y_pred = rfc.predict(x_test)

clrp = classification_report(y_test, y_pred)
print(clrp)

              precision    recall  f1-score   support

           0       0.98      0.99      0.98       199
           1       0.99      0.97      0.98       651
           2       0.96      0.98      0.97       350

    accuracy                           0.98      1200
   macro avg       0.98      0.98      0.98      1200
weighted avg       0.98      0.98      0.98      1200



## **Support Vector Classifier**

In [291]:
from sklearn.svm import SVC

svc = SVC(kernel="linear", C=1.0)

svc.fit(x_train, y_train)

y_pred = svc.predict(x_test)

classr = classification_report(y_test, y_pred)
print(classr)


              precision    recall  f1-score   support

           0       0.91      0.87      0.89       199
           1       0.93      0.94      0.93       651
           2       0.94      0.93      0.93       350

    accuracy                           0.93      1200
   macro avg       0.92      0.92      0.92      1200
weighted avg       0.93      0.93      0.93      1200



## **Comaparison**

In [293]:
from tabulate import tabulate

columns = ["Model", "Accuracy", "Precision", "Recall", "F1_score"]

result = [["Logistic Regression", 0.90, 0.87, 0.78, 0.82],
          ["Decision Tree Classifier", 0.97, 0.97, 0.97, 0.97],
          ["Random Forest Classifier", 0.98, 0.98, 0.99, 0.98],
          ["Support Vector Classifier", 0.93, 0.91, 0.87, 0.89]
          ]

table = tabulate(result, headers = columns, tablefmt="grid", floatfmt='.2f')
print(table)

+---------------------------+------------+-------------+----------+------------+
| Model                     |   Accuracy |   Precision |   Recall |   F1_score |
+===========================+============+=============+==========+============+
| Logistic Regression       |       0.90 |        0.87 |     0.78 |       0.82 |
+---------------------------+------------+-------------+----------+------------+
| Decision Tree Classifier  |       0.97 |        0.97 |     0.97 |       0.97 |
+---------------------------+------------+-------------+----------+------------+
| Random Forest Classifier  |       0.98 |        0.98 |     0.99 |       0.98 |
+---------------------------+------------+-------------+----------+------------+
| Support Vector Classifier |       0.93 |        0.91 |     0.87 |       0.89 |
+---------------------------+------------+-------------+----------+------------+


# **Regression Algorithms**

In [296]:
df.sample(10)

,age,daily_screen_time,social_media_hours,study_hours,sleep_hours,notifications_per_day,focus_score,addiction_level,productivity_score
1813,0.875000,0.212212,0.125000,0.11875,0.332,0.637993,1.000000,0,0.1288
4071,0.583333,0.641642,0.327000,0.57500,0.436,0.311828,1.000000,1,0.4539
1131,0.541667,0.473473,0.192000,0.67875,0.184,0.161290,1.000000,1,0.5483
2097,0.166667,0.087087,0.348591,0.40125,0.042,0.960573,1.000000,0,0.3787
2285,0.250000,0.032032,0.045000,0.43250,0.332,0.824373,1.000000,0,0.4805
4957,0.125000,0.329329,0.312000,0.28125,0.120,0.229391,1.000000,1,0.0000
2484,0.666667,0.921922,0.632000,0.06375,0.186,0.301075,0.713743,2,0.0000
4431,0.541667,0.259259,0.251000,0.29375,0.860,0.831541,1.000000,1,0.3529
5031,0.541667,0.913914,0.375000,0.14750,0.170,0.863799,0.565102,1,0.2409
433,0.416667,0.587588,0.641000,0.34000,0.732,0.132616,1.000000,2,0.0952


In [297]:
x = df.drop("productivity_score", axis=1)
y= df["productivity_score"]

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2, random_state=42)

## **LInear Regression**

In [305]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

lr = LinearRegression()

lr.fit(x_train, y_train)

y_pred = lr.predict(x_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(mae)
print(mse)
print(rmse)
print(r2)

0.08506650639554562
0.01239830883386489
0.11134769343756022
0.8317685760231144


## **Decision Tree Regressor**


In [310]:
from sklearn.tree import DecisionTreeRegressor

dtr = DecisionTreeRegressor()

dtr.fit(x_train, y_train)

y_pred = dtr.predict(x_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(mae)
print(mse)
print(rmse)
print(r2)


0.11108549999999999
0.023348092992748946
0.15280082785361127
0.6831920398218964


## **Random Forest Regressor**

In [315]:
from sklearn.ensemble import RandomForestRegressor

rfr = RandomForestRegressor()

rfr.fit(x_train, y_train)

y_pred = rfr.predict(x_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(mae)
print(mse)
print(rmse)
print(r2)


0.08216441711054422
0.012048341494904103
0.10976493745684049
0.836517248166041


## **Support Vector Regressor**

In [320]:
from sklearn.svm import SVR

svr = SVR()

svr.fit(x_train, y_train)

y_pred = svr.predict(x_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(mae)
print(mse)
print(rmse)
print(r2)


0.08186887443595872
0.011386507696261884
0.10670758031303064
0.845497605396496


## **Comparison**

In [321]:
columns = ["Algorithm", "Mean Absolute Error", "Mean Squared Error", "Root Mean Squared Error", "R2_score"]

results = [["Linear Regression", 0.085, 0.012, 0.111, 0.831],
           ["Decision Tree Regressor", 0.111, 0.023, 0.152, 0.68],
           ["Random Forest Regressor", 0.082, 0.012, 0.109, 0.836],
           ["Support Vector Regressor", 0.81, 0.011, 0.106, 0.845]]

table = tabulate(results, headers = columns, tablefmt = 'grid', floatfmt='.2f')
print(table)

+--------------------------+-----------------------+----------------------+---------------------------+------------+
| Algorithm                |   Mean Absolute Error |   Mean Squared Error |   Root Mean Squared Error |   R2_score |
+==========================+=======================+======================+===========================+============+
| Linear Regression        |                  0.09 |                 0.01 |                      0.11 |       0.83 |
+--------------------------+-----------------------+----------------------+---------------------------+------------+
| Decision Tree Regressor  |                  0.11 |                 0.02 |                      0.15 |       0.68 |
+--------------------------+-----------------------+----------------------+---------------------------+------------+
| Random Forest Regressor  |                  0.08 |                 0.01 |                      0.11 |       0.84 |
+--------------------------+-----------------------+------------